# 07 — Inferencia + Score de Confianza

> **Objetivo.** Servir el modelo en una función reusable que devuelve, además de la corrección, una **medida de confianza** derivada de los logits.

## ¿Por qué necesitamos confianza?

Un modelo seq2seq siempre devuelve *algo*. Sin un score:
- No podemos distinguir "corrección segura" de "adivinanza".
- No podemos enviar a revisión humana solo los casos dudosos.
- No podemos decidir cuándo NO mostrar la sugerencia.

## Cómo derivarla — log-probs por token

`model.generate(..., output_scores=True, return_dict_in_generate=True)` devuelve, además de los `sequences`, un tensor `scores[t]` con los **logits del token generado en el paso t** (uno por paso de decodificación).

El flujo:

```
logits[t]               →  log_softmax  →  log_probs[t]
log_prob_token_elegido  =  log_probs[t][token_id_elegido]
confidence              =  exp(mean(log_prob_token_elegido))
```

La exponenciación del log-prob promedio da una **probabilidad geométrica media** ∈ (0,1] — interpretable como "probabilidad por token".

## Heurísticas adicionales

1. Si `output == input` → confianza alta de **"oración ya correcta"**.
2. Si `confidence < UMBRAL` → marcar para revisión humana.
3. Si edit-distance grande con respecto al input → revisar (puede ser hallucinación).

In [ ]:
# === CONFIG ===
NUM_BEAMS       = 4
MAX_NEW_TOKENS  = 128
LOW_CONF_THRESH = 0.85   # bajo este score → marcar para revisión
# ==============

import os, json, math, torch
from pathlib import Path
from transformers import AutoTokenizer, T5ForConditionalGeneration

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

ROOT      = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_DIR = ROOT / "artifacts" / "t5-gec-mac" / "final"

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = T5ForConditionalGeneration.from_pretrained(str(MODEL_DIR)).to(device)
model.eval()

## 1. Función de inferencia con confianza

In [ ]:
PREFIX = "grammar: "

def correct_with_confidence(text: str, num_beams: int = NUM_BEAMS):
    """Devuelve {original, suggestion, confidence, edited, decision}."""
    enc = tokenizer(PREFIX + text, return_tensors="pt", truncation=True, max_length=128).to(device)

    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=num_beams,
            output_scores=True,
            return_dict_in_generate=True,
            early_stopping=True,
        )

    seq = out.sequences[0]
    suggestion = tokenizer.decode(seq, skip_special_tokens=True).strip()

    # `compute_transition_scores` mapea scores+sequences → log-probs por token (post beam-search)
    transition = model.compute_transition_scores(
        out.sequences, out.scores, out.beam_indices, normalize_logits=True
    )[0]

    # Excluir tokens posteriores al EOS (ya no informan)
    valid = transition[~torch.isinf(transition)]
    if len(valid) == 0:
        confidence = 0.0
    else:
        confidence = float(math.exp(valid.mean().item()))

    edited = suggestion.lower().strip() != text.lower().strip()

    if not edited:
        decision = "no_change_needed"
    elif confidence >= LOW_CONF_THRESH:
        decision = "apply"
    else:
        decision = "review"

    return {
        "original": text,
        "suggestion": suggestion,
        "confidence": round(confidence, 4),
        "edited": edited,
        "decision": decision,
    }

## 2. Probar con ejemplos diversos

In [ ]:
examples = [
    # error claro
    "He go to store yesterday.",
    "I has a apple.",
    "She don't like coffee.",
    "Many child play in the park.",
    "This depends of the result.",

    # ya correcta
    "This is a perfectly fine sentence.",
    "Machine learning is a subfield of artificial intelligence.",

    # ambigua / borderline
    "He read a book yesterday.",   # válido como pasado
    "The weather is nice today.",

    # ortografía
    "I recieved your email.",
    "This is definately wrong.",
]

results = [correct_with_confidence(t) for t in examples]

for r in results:
    print(f"[{r['decision']:<18}]  conf={r['confidence']:.3f}")
    print(f"  IN : {r['original']}")
    print(f"  OUT: {r['suggestion']}")
    print()

## 3. Inferencia por lote (eficiente)

Para servir muchas oraciones a la vez, batch helper. Mantiene el cálculo de confianza pero amortiza overhead.

In [ ]:
def correct_batch(texts, batch_size=8, num_beams=NUM_BEAMS):
    out = []
    for i in range(0, len(texts), batch_size):
        chunk = texts[i:i+batch_size]
        enc = tokenizer([PREFIX + t for t in chunk],
                        return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            res = model.generate(
                **enc,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=num_beams,
                output_scores=True,
                return_dict_in_generate=True,
                early_stopping=True,
            )
        decoded = tokenizer.batch_decode(res.sequences, skip_special_tokens=True)
        scores = model.compute_transition_scores(res.sequences, res.scores, res.beam_indices, normalize_logits=True)

        for j, sugg in enumerate(decoded):
            valid = scores[j][~torch.isinf(scores[j])]
            conf = float(math.exp(valid.mean().item())) if len(valid) > 0 else 0.0
            edited = sugg.strip().lower() != chunk[j].strip().lower()
            decision = "no_change_needed" if not edited else ("apply" if conf >= LOW_CONF_THRESH else "review")
            out.append({
                "original": chunk[j],
                "suggestion": sugg.strip(),
                "confidence": round(conf, 4),
                "edited": edited,
                "decision": decision,
            })
    return out

batch_results = correct_batch(examples, batch_size=4)
for r in batch_results[:3]:
    print(r)

## 4. Calibración del umbral

¿`LOW_CONF_THRESH=0.85` es el valor correcto? Lo justificamos empíricamente sobre el eval set: para cada nivel de threshold computamos precision (de los aceptados, cuántos coinciden con la referencia).

In [ ]:
from datasets import load_from_disk
import numpy as np
import matplotlib.pyplot as plt

SPLITS = ROOT / "artifacts" / "coedit_gec_splits"
eval_set = load_from_disk(str(SPLITS))["test"]

N = min(500, len(eval_set))   # muestra para velocidad
sample_src = list(eval_set["src"][:N])
sample_ref = list(eval_set["tgt"][:N])

preds = correct_batch(sample_src, batch_size=8)

In [ ]:
def norm(s): return " ".join(s.lower().strip().split())

rows = []
for p, ref in zip(preds, sample_ref):
    rows.append((p["confidence"], norm(p["suggestion"]) == norm(ref), p["edited"]))

import pandas as pd
df = pd.DataFrame(rows, columns=["confidence", "correct", "edited"])

thresholds = np.linspace(0.5, 1.0, 21)
stats = []
for t in thresholds:
    accepted = df[df["confidence"] >= t]
    if len(accepted) == 0:
        prec = np.nan
    else:
        prec = accepted["correct"].mean()
    coverage = len(accepted) / len(df)
    stats.append({"threshold": t, "precision": prec, "coverage": coverage})
stats_df = pd.DataFrame(stats)
print(stats_df.round(3).to_string(index=False))

In [ ]:
fig, ax1 = plt.subplots(figsize=(8,4))
ax1.plot(stats_df["threshold"], stats_df["precision"], color="#F06105", marker="o", label="Precision")
ax1.set_xlabel("Confidence threshold")
ax1.set_ylabel("Precision", color="#F06105")
ax2 = ax1.twinx()
ax2.plot(stats_df["threshold"], stats_df["coverage"], color="#3578A0", marker="s", label="Coverage")
ax2.set_ylabel("Coverage", color="#3578A0")
plt.title("Precision/Coverage vs threshold de confianza")
plt.tight_layout()
plt.show()

**Lectura.** Sube el threshold → menos sugerencias aplicadas (coverage ↓) pero las que se aplican son más correctas (precision ↑). Elegimos el threshold que cumpla el SLA del producto. Para revisión humana asistida, threshold alto + revisión manual del resto.

## 5. Persistir muestra de inferencias

In [ ]:
OUT = ROOT / "artifacts" / "inference_samples.json"
OUT.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print("Guardado en:", OUT)

## Resumen

- API `correct_with_confidence(text)` devuelve sugerencia + score interpretable (∈ 0..1).
- Heurística de decisión `apply / review / no_change_needed` consumible por capa de servicio.
- Curvas precision/coverage para calibrar el umbral según SLA.

**Siguiente (opcional) → `08_lora_peft.ipynb`** — escalar a `t5-base` con LoRA.
